# Lab 08-09 · Arithmetic Circuits

Companion to chapter [08-09](../../chapters/08-09-arithmetic-circuits.md). Handout: [README.md](README.md).

Every circuit here is built from its gates or cells and checked against Python's own integer arithmetic. Delays use the chapter's unit-delay model: every two-input gate costs 1 unit, a prefix operator 2 units on its $G$ output and 1 on its $P$ output, and the carry-in is folded into bit 0's generate (ready at 3). Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import random
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

rng = random.Random(809)

## Part 1 · Prefix adders (the chapter's guided exercise)

A prefix network is a list of levels; each level is a list of operators $(i, j)$: position $i$ combines its current span with position $j$'s span, which must end just below it.

**🔮 Predict:** going from 16 to 64 bits, by what factor do a Kogge–Stone adder's delay and its operator count grow?

In [2]:
def ripple(n):
    return [[(i, i - 1)] for i in range(1, n)]


def kogge_stone(n):
    levels, d = [], 1
    while d < n:
        levels.append([(i, i - d) for i in range(d, n)])
        d *= 2
    return levels


def sklansky(n):
    levels, d = [], 1
    while d < n:
        levels.append([(i, (i // d) * d - 1) for i in range(n) if (i // d) % 2 == 1])
        d *= 2
    return levels


def brent_kung(n):
    levels, d = [], 1
    while d < n:                                   # up-sweep: a binary tree of spans
        levels.append([(i, i - d) for i in range(2 * d - 1, n, 2 * d)])
        d *= 2
    d //= 4
    while d >= 1:                                  # down-sweep: fill in the other prefixes
        lv = [(i, i - d) for i in range(3 * d - 1, n, 2 * d)]
        if lv:
            levels.append(lv)
        d //= 2
    return levels


def simulate(levels, n, a, b, cin):
    # returns (a + b + cin computed by the network, delay to the slowest sum bit)
    A = [(a >> i) & 1 for i in range(n)]
    B = [(b >> i) & 1 for i in range(n)]
    P = [x ^ y for x, y in zip(A, B)]
    g = [x & y for x, y in zip(A, B)]
    g[0] |= P[0] & cin
    p = P[:]
    lo = list(range(n))                            # lowest bit each position's span covers
    tg, tp = [1] * n, [1] * n
    tg[0] = 3
    for lv in levels:
        ng, np_, ntg, ntp, nlo = g[:], p[:], tg[:], tp[:], lo[:]
        for i, j in lv:
            assert lo[i] == j + 1, "spans must be adjacent"
            ng[i] = g[i] | (p[i] & g[j])
            np_[i] = p[i] & p[j]
            ntg[i] = max(tg[i], max(tp[i], tg[j]) + 1) + 1
            ntp[i] = max(tp[i], tp[j]) + 1
            nlo[i] = lo[j]
        g, p, tg, tp, lo = ng, np_, ntg, ntp, nlo
    assert all(x == 0 for x in lo), "every position must reach bit 0"
    s = sum((P[i] ^ (g[i - 1] if i else cin)) << i for i in range(n)) | (g[n - 1] << n)
    return s, max(max(1, tg[i - 1] if i else 0) + 1 for i in range(n))


NETS = {"ripple": ripple, "Brent-Kung": brent_kung, "Sklansky": sklansky, "Kogge-Stone": kogge_stone}
table = {}
for name, f in NETS.items():
    for n in (4, 8, 16, 32, 64):
        L = f(n)
        for _ in range(300):
            a, b, c = rng.getrandbits(n), rng.getrandbits(n), rng.getrandbits(1)
            assert simulate(L, n, a, b, c)[0] == a + b + c
        fan = max((sum(1 for i, j in lv if j == k) for lv in L for k in range(n)), default=0)
        table[name, n] = (sum(map(len, L)), len(L), fan, simulate(L, n, 0, 0, 0)[1])
print(f"{'network':12s} {'n':>3s} {'ops':>5s} {'levels':>7s} {'fan-out':>8s} {'delay':>6s}")
for (name, n), (ops, lv, fan, d) in table.items():
    print(f"{name:12s} {n:3d} {ops:5d} {lv:7d} {fan:8d} {d:6d}")
ks16, ks64 = table["Kogge-Stone", 16], table["Kogge-Stone", 64]
print(f"\nKogge-Stone, 16 -> 64 bits: delay x{ks64[3] / ks16[3]:.2f}, operators x{ks64[0] / ks16[0]:.2f}")
assert (ks16[0], ks16[3], ks64[0], ks64[3]) == (49, 10, 321, 14)

network        n   ops  levels  fan-out  delay
ripple         4     3       3        1      8
ripple         8     7       7        1     16
ripple        16    15      15        1     32
ripple        32    31      31        1     64
ripple        64    63      63        1    128
Brent-Kung     4     4       3        1      8
Brent-Kung     8    11       5        1     12
Brent-Kung    16    26       7        1     16
Brent-Kung    32    57       9        1     20
Brent-Kung    64   120      11        1     24
Sklansky       4     4       2        2      8
Sklansky       8    12       3        4     10
Sklansky      16    32       4        8     12
Sklansky      32    80       5       16     14
Sklansky      64   192       6       32     16
Kogge-Stone    4     5       2        1      6
Kogge-Stone    8    17       3        1      8
Kogge-Stone   16    49       4        1     10
Kogge-Stone   32   129       5        1     12
Kogge-Stone   64   321       6        1     14

Kogge-Stone,

Four times the width costs Kogge–Stone only 40% more delay (two more levels) but 6.6 times the operators: the operator count grows as $n\log_2 n$. Every one of those operators switches, leaks and needs wires, which is why 64-bit adders in practice often use sparser networks (computing every second or fourth carry with a prefix tree and the rest with short ripple or select stages) that give up a level or so of delay for far fewer operators.

In [3]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
widths = [4, 8, 16, 32, 64]
for name in NETS:
    ax[0].plot(widths, [table[name, n][3] for n in widths], "o-", label=name)
    ax[1].plot(widths, [table[name, n][0] for n in widths], "o-", label=name)
for a_, lab in zip(ax, ["delay (unit gate delays)", "prefix operators"]):
    a_.set_xscale("log", base=2)
    a_.set_xticks(widths)
    a_.set_xticklabels([str(w) for w in widths])
    a_.set_xlabel("width (bits)")
    a_.set_ylabel(lab)
ax[0].legend()
ax[0].set_title("Delay")
ax[1].set_title("Hardware")
plt.tight_layout()
plt.show()

## Part 2 · A Dadda multiplier, bit by bit

The cell below builds the partial-product matrix of an $n \times n$ unsigned multiplier, reduces it level by level to the Dadda target heights with full and half adders, and adds the final two rows. It checks every product against Python's `*` and counts the adders.

In [4]:
def dadda_targets(n):
    d = [2]
    while d[-1] < n:
        d.append(d[-1] * 3 // 2)
    return [t for t in reversed(d) if t < n]


def dadda_multiply(a, b, n, count=None):
    cols = [[((a >> i) & 1) & ((b >> j) & 1) for i in range(n) for j in range(n) if i + j == c]
            for c in range(2 * n)]
    fa = ha = 0
    for t in dadda_targets(n):
        new = [[] for _ in range(2 * n + 1)]
        for c in range(2 * n):
            bits = cols[c]
            h = len(bits) + len(new[c])            # new[c] already holds carries from column c - 1
            while h > t:
                if h - t >= 2 and len(bits) >= 3:  # full adder: 3 bits in, sum here, carry to c + 1
                    x, y, z = bits.pop(), bits.pop(), bits.pop()
                    new[c].append(x ^ y ^ z)
                    new[c + 1].append((x & y) | (x & z) | (y & z))
                    fa += 1
                    h -= 2
                else:                              # half adder: 2 bits in
                    x, y = bits.pop(), bits.pop()
                    new[c].append(x ^ y)
                    new[c + 1].append(x & y)
                    ha += 1
                    h -= 1
            new[c].extend(bits)
        cols = new[:2 * n]
    if count is not None:
        count.update(fa=fa, ha=ha, final=sum(1 for col in cols if len(col) == 2))
    return sum(sum(col) << c for c, col in enumerate(cols))


print(" n  levels  full adders  n^2-4n+3  half adders  final adder bits")
for n in (4, 8, 12, 16, 24, 32):
    for _ in range(200):
        a, b = rng.getrandbits(n), rng.getrandbits(n)
        assert dadda_multiply(a, b, n) == a * b
    cnt = {}
    dadda_multiply(1, 1, n, cnt)
    print(f"{n:2d}  {len(dadda_targets(n)):6d}  {cnt['fa']:11d}  {n * n - 4 * n + 3:8d}  {cnt['ha']:11d}  {cnt['final']:16d}")
    assert cnt["fa"] == n * n - 4 * n + 3 and cnt["ha"] == n - 1

 n  levels  full adders  n^2-4n+3  half adders  final adder bits
 4       2            3         3            3                 6
 8       4           35        35            7                14
12       5           99        99           11                22


16       6          195       195           15                30


24       7          483       483           23                46


32       8          899       899           31                62


The full-adder count matches the chapter's bit-counting argument ($n^2 - 4n + 3$) for every width, and the levels grow very slowly: 32 × 32 needs only 8.

## Part 3 · Booth recoding of every 8-bit multiplier

**🔮 Predict:** averaged over all 256 signed 8-bit multipliers, how many of the 4 radix-4 Booth digits are nonzero? How many 1 bits does an 8-bit number have on average?

In [5]:
def booth_digits(B, n=8):
    bits = [(B >> i) & 1 for i in range(n)]          # two's-complement bits of B
    padded = [0] + bits                              # b_{-1} = 0 in front
    return [-2 * padded[2 * k + 2] + padded[2 * k + 1] + padded[2 * k] for k in range(n // 2)]


nonzero, ones = [], []
for B in range(-128, 128):
    d = booth_digits(B & 0xFF)
    assert sum(dk * 4 ** k for k, dk in enumerate(d)) == B
    assert all(-2 <= dk <= 2 for dk in d)
    nonzero.append(sum(dk != 0 for dk in d))
    ones.append(bin(B & 0xFF).count("1"))
print("Booth digits of 122:", booth_digits(122), "  of -99:", booth_digits(-99 & 0xFF))
print(f"average nonzero Booth digits: {np.mean(nonzero):.3f} of 4;  average 1 bits: {np.mean(ones):.3f} of 8")
print("distribution of nonzero digits:", {k: nonzero.count(k) for k in range(5)})
assert booth_digits(122) == [-2, -1, 0, 2]

Booth digits of 122: [-2, -1, 0, 2]   of -99: [1, -1, 2, -2]
average nonzero Booth digits: 3.000 of 4;  average 1 bits: 4.000 of 8
distribution of nonzero digits: {0: 1, 1: 12, 2: 54, 3: 108, 4: 81}


A digit is zero exactly when its triplet is 000 or 111. Over all 256 multipliers, each of the three upper triplets takes all 8 patterns equally often, so each of those digits is zero a quarter of the time; the lowest triplet ends in the fixed $b_{-1} = 0$, so it takes only the 4 patterns $xy0$ and is zero only for 000, again a quarter of the time. On average $4 \times 3/4 = 3$ of the 4 digits are nonzero, against 4 of the 8 bits for plain binary. Booth's real saving is the guaranteed halving of the number of rows, not the zeros.

## Part 4 · Fixed point: rounding bias and SQNR

Multiply random Q1.15 numbers, convert the Q2.30 products back to Q1.15 by truncation and by rounding, and compare with the exact product. Then quantize a full-scale sine to $b$ bits and measure the SQNR.

In [6]:
nrng = np.random.default_rng(809)
a = nrng.integers(-32768, 32768, 200_000)
b = nrng.integers(-32768, 32768, 200_000)
prod = a * b                                         # exact Q2.30 integers
exact = prod / 2 ** 15                               # exact value in Q1.15 LSBs
trunc = prod >> 15                                   # arithmetic shift: rounds toward -inf
rnd = (prod + 2 ** 14) >> 15                         # round half up
print(f"mean error, truncation: {np.mean(trunc - exact):+.4f} LSB   rounding: {np.mean(rnd - exact):+.6f} LSB")


def sqnr_db(bits, n=2 ** 16):
    t = np.arange(n)
    x = np.sin(2 * np.pi * 1031 / n * t + 0.3)       # full scale, 1031 cycles in the record
    q = np.round(x * (2 ** (bits - 1) - 1)) / (2 ** (bits - 1) - 1)
    return 10 * np.log10(np.mean(x ** 2) / np.mean((x - q) ** 2))


for bits in (8, 12, 16):
    print(f"{bits:2d} bits: measured SQNR {sqnr_db(bits):6.2f} dB   formula 6.02b + 1.76 = {6.02 * bits + 1.76:6.2f} dB")

mean error, truncation: -0.5008 LSB   rounding: +0.000127 LSB
 8 bits: measured SQNR  49.95 dB   formula 6.02b + 1.76 =  49.92 dB
12 bits: measured SQNR  74.03 dB   formula 6.02b + 1.76 =  74.00 dB
16 bits: measured SQNR  98.07 dB   formula 6.02b + 1.76 =  98.08 dB


Truncation's bias of almost $-\tfrac12$ LSB is what Problem 08-09.13 computed exactly; rounding removes it. The measured SQNR agrees with $6.02b + 1.76$ dB to within a few hundredths of a dB. The small differences, in both directions, come from the sine's amplitude being $2^{b-1} - 1$ codes rather than the $2^{b-1}$ the formula assumes, and from the quantization error not being exactly uniform for a finite record.

**Verilog.** Now run `arith_tb.v` (`python tools/run_hdl.py` from the repository root). The Q1.15 multiplier there uses the same round-half-up rule as this notebook, with saturation for the single overflow case $(-1) \times (-1)$.